# 2차 — LightGBM LambdaRank 손실함수 실험

## 1차 결과 (group_size=100)
- OOF AUC **0.73599** — 기존 binary 단일시드(0.7400~0.74053)보다 분명히 낮음
- fold간 표준편차 **0.00130** — 기존 모델 배깅 변동성(0.00011)의 약 12배
- → 그룹을 랜덤 셔플로 인공 구성하는 방식 자체가 노이즈원일 가능성 → **group_size를 키워서 재확인**

## 핵심 아이디어
- `objective='lambdarank'` + `metric='auc'`로 학습 중에도 실제 평가지표를 직접 모니터링
- LightGBM lambdarank는 query(그룹) 하나당 행 수가 10,000개를 못 넘는 하드캡이 있음(pairwise 비교가 O(n²)이라 안전장치) → fold 전체를 한 그룹으로 묶지 않고, 셔플 후 작은 인공 그룹으로 잘라서 그 안에서만 비교
- **이번 버전은 group_size 여러 값을 한 번에 비교**해서, 1차 결과(100)가 너무 작아서 노이즈가 컸던 건지 확인
- lambdarank 출력은 보정된 확률이 아니라 임의 스케일의 순위 점수이므로, fold별로 즉시 rank-정규화([0,1] 백분위) 해서 합침

## 전처리 / 파라미터 출처
- 전처리는 `2차_lgbm_robust_multiseed_search` 노트북의 실제 코드 그대로 (TARGET_COL="임신 성공 여부", DEAD_COLS 2개 드롭, `is_DI`/`froze_embryo` 파생, 결측치 처리, LabelEncoder — train+test 합쳐서 fit)
- 시작 파라미터는 멀티시드 탐색에서 채택된 `lgbm_robust_best_params.json`

---
**저장 위치:** `experiment_history/2차/2차_lgbm_lambdarank.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score


In [2]:
# 노트북 위치: experiment_history/2차/ (멀티시드 탐색 노트북과 동일 폴더)
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")              # blend_cache, *_best_params.json 등 공유 자원
SUBMIT_DIR = Path("../submit file")  # 실험 중 생성되지만 실제 제출은 안 한 CSV

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]

# 멀티시드 탐색에서 채택된 파라미터를 기본값으로. 없으면 기존 best_params.json으로 자동 폴백
ROBUST_PARAMS_PATH = SHARED_DIR / "lgbm_robust_best_params.json"
FALLBACK_PARAMS_PATH = SHARED_DIR / "best_params.json"

N_SPLITS = 5
SEED = 42
EARLY_STOPPING_ROUNDS = 100
N_BOOST_ROUND = 3000  # 충분히 크게, early stopping이 실제 라운드 수를 결정

# group_size 그리드: 1차(100)가 너무 작아서 노이즈가 컸는지 확인하기 위해 여러 값 비교
# 비용은 거의 group_size에 선형 비례 (총 pairwise 비교 ≈ n_rows * group_size) — 5-fold 전체가
# group_size=100일 때 13.8초였으니 1000도 충분히 빠를 것으로 예상
CANDIDATE_GROUP_SIZES = [50, 100, 300, 1000]


## 1. 데이터 로드 + 전처리 (멀티시드 탐색 노트북과 동일 로직, train+test 둘 다 처리)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
test_raw = test_raw_full.drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df_train = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))
df_test = fill_na(base_features(test_raw.copy()).drop(columns=DEAD_COLS))

cat_cols = df_train.select_dtypes(include=["object"]).columns.tolist()

# train+test 합쳐서 LabelEncoder fit (카테고리 불일치로 인한 크래시 방지 — xgboost에서 겪은 것과 동일 패턴)
df_train_le = df_train.copy()
df_test_le = df_test.copy()
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([df_train_le[col], df_test_le[col]], axis=0).astype(str)
    le.fit(combined)
    df_train_le[col] = le.transform(df_train_le[col].astype(str))
    df_test_le[col] = le.transform(df_test_le[col].astype(str))

y = df_train_le[TARGET_COL].values.astype(np.float32)
X = df_train_le.drop(columns=[TARGET_COL])
X_test = df_test_le.copy()

print(f"전처리 완료: train {X.shape}, test {X_test.shape}")


/var/folders/s7/xbnkmtj92j5_ly_bxspy44f40000gn/T/ipykernel_94740/2307222159.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns
/var/folders/s7/xbnkmtj92j5_ly_bxspy44f40000gn/T/ipykernel_94740/2307222159.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://p

전처리 완료: train (256351, 67), test (90067, 67)


## 2. CV 설정 + 시작 파라미터 로드

In [4]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
print(f"{N_SPLITS}-fold 분할 완료 (seed={SEED})")


5-fold 분할 완료 (seed=42)


In [5]:
def load_base_params(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        loaded = json.load(f)
    return loaded.get("best_params", loaded) if isinstance(loaded, dict) else loaded

params_path = ROBUST_PARAMS_PATH if ROBUST_PARAMS_PATH.exists() else FALLBACK_PARAMS_PATH
base_params = load_base_params(params_path)
print(f"파라미터 출처: {params_path}")

# n_estimators는 num_boost_round(함수 인자)와 역할이 겹쳐서 분리, 나머지 sklearn식 이름은
# LightGBM이 공식 alias로 인식하므로 그대로 둠 (subsample→bagging_fraction 등)
base_params.pop("n_estimators", None)

# binary objective 전용 키 제거 (혹시 들어있다면 — 어차피 효과 없었던 레버였음)
for k in ["objective", "metric", "is_unbalance", "scale_pos_weight"]:
    base_params.pop(k, None)

params = {
    **base_params,
    "objective": "lambdarank",
    "metric": "auc",        # objective와 무관하게 실제 평가지표를 직접 모니터링
    "label_gain": [0, 1],   # 이진 레이블(0/1)이므로 명시
    "random_state": SEED,
    "verbosity": -1,
}

print(json.dumps(params, indent=2, ensure_ascii=False))


파라미터 출처: ../lgbm_robust_best_params.json
{
  "learning_rate": 0.03735839746471122,
  "num_leaves": 253,
  "max_depth": 5,
  "min_child_samples": 41,
  "subsample": 0.699825092698368,
  "colsample_bytree": 0.7084506083750325,
  "reg_alpha": 3.9529019873195606,
  "reg_lambda": 5.86734589907409,
  "min_split_gain": 0.5142456412311088,
  "objective": "lambdarank",
  "metric": "auc",
  "label_gain": [
    0,
    1
  ],
  "random_state": 42,
  "verbosity": -1
}


## 3. fold별 학습 함수 정의

In [6]:
def rank_normalize(scores: np.ndarray) -> np.ndarray:
    """fold마다 임의 스케일인 lambdarank 점수를 [0,1] 백분위로 정규화"""
    return rankdata(scores) / len(scores)


def make_group_sizes(n: int, group_size: int) -> list:
    """n개 행을 group_size 단위로 잘라서 그룹 크기 리스트를 만듦 (마지막 그룹은 나머지)"""
    sizes = [group_size] * (n // group_size)
    remainder = n % group_size
    if remainder:
        sizes.append(remainder)
    return sizes


def shuffle_rows(X_part: pd.DataFrame, y_part: np.ndarray, seed: int):
    """인공 그룹을 만들기 전에 행 순서를 섞음 (그룹마다 0/1이 적절히 섞이도록)"""
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(X_part))
    return X_part.iloc[order].reset_index(drop=True), y_part[order]


def run_lambdarank_cv(group_size: int, verbose: bool = False):
    oof_ = np.zeros(len(X))
    test_fold_preds_ = []
    fold_aucs_ = []

    start = time.time()
    for fold, (tr_idx, val_idx) in enumerate(folds):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        X_tr_shuf, y_tr_shuf = shuffle_rows(X_tr, y_tr, seed=SEED + fold)
        X_val_shuf, y_val_shuf = shuffle_rows(X_val, y_val, seed=SEED + fold + 1000)

        train_group = make_group_sizes(len(X_tr_shuf), group_size)
        valid_group = make_group_sizes(len(X_val_shuf), group_size)

        train_set = lgb.Dataset(X_tr_shuf, label=y_tr_shuf, group=train_group)
        valid_set = lgb.Dataset(X_val_shuf, label=y_val_shuf, group=valid_group, reference=train_set)

        callbacks = [lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)]
        if verbose:
            callbacks.append(lgb.log_evaluation(100))

        model = lgb.train(
            params,
            train_set,
            num_boost_round=N_BOOST_ROUND,
            valid_sets=[valid_set],
            callbacks=callbacks,
        )

        # 예측은 원래 순서 그대로의 X_val/X_test에 대해 수행 — group은 학습시 그래디언트 계산에만
        # 쓰이고, predict()는 row 순서/그룹과 무관하게 각 행의 점수를 그대로 출력함
        val_raw = model.predict(X_val, num_iteration=model.best_iteration)
        fold_auc = roc_auc_score(y_val, val_raw)
        fold_aucs_.append(fold_auc)
        oof_[val_idx] = rank_normalize(val_raw)

        test_raw = model.predict(X_test, num_iteration=model.best_iteration)
        test_fold_preds_.append(rank_normalize(test_raw))

    elapsed_ = time.time() - start
    return oof_, test_fold_preds_, fold_aucs_, elapsed_


## 4. group_size 그리드서치 (1차 결과가 노이즈였는지 확인)

In [7]:
results = {}
print(f"{'group_size':>10} | {'OOF AUC':>8} | {'fold_std':>9} | {'time(s)':>8}")
print("-" * 45)
for gs in CANDIDATE_GROUP_SIZES:
    oof_gs, test_preds_gs, fold_aucs_gs, elapsed = run_lambdarank_cv(gs)
    oof_auc_gs = roc_auc_score(y, oof_gs)
    fold_std_gs = float(np.std(fold_aucs_gs))
    results[gs] = {
        "oof": oof_gs,
        "test_fold_preds": test_preds_gs,
        "fold_aucs": fold_aucs_gs,
        "oof_auc": oof_auc_gs,
        "fold_std": fold_std_gs,
        "elapsed": elapsed,
    }
    print(f"{gs:>10} | {oof_auc_gs:>8.5f} | {fold_std_gs:>9.5f} | {elapsed:>8.1f}")

best_gs = max(results, key=lambda g: results[g]["oof_auc"])
print(f"\n채택: group_size={best_gs}  (OOF AUC={results[best_gs]['oof_auc']:.5f})")
print()
print("비교 기준:")
print("  기존 binary LightGBM 단일시드 OOF: 0.7400 ~ 0.74053")
print("  멀티시드 배깅 OOF: 0.74036 (기존) / 0.74037 (견고탐색)")
print("  팀 블렌딩 OOF: 0.74071")


group_size |  OOF AUC |  fold_std |  time(s)
---------------------------------------------
        50 |  0.73748 |   0.00147 |     16.9
       100 |  0.73599 |   0.00130 |     12.8
       300 |  0.73237 |   0.00111 |     10.5
      1000 |  0.72860 |   0.00135 |      8.8

채택: group_size=50  (OOF AUC=0.73748)

비교 기준:
  기존 binary LightGBM 단일시드 OOF: 0.7400 ~ 0.74053
  멀티시드 배깅 OOF: 0.74036 (기존) / 0.74037 (견고탐색)
  팀 블렌딩 OOF: 0.74071


## 5. 최적 group_size 결과로 확정

In [8]:
oof = results[best_gs]["oof"]
test_fold_preds = results[best_gs]["test_fold_preds"]
fold_aucs = results[best_gs]["fold_aucs"]
oof_auc = results[best_gs]["oof_auc"]

print(f"채택된 group_size={best_gs}")
print(f"Fold별 AUC: {[round(a, 5) for a in fold_aucs]}")
print(f"Fold AUC 평균±표준편차: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}")
print(f"전체 OOF AUC (rank-정규화 기준): {oof_auc:.5f}")


채택된 group_size=50
Fold별 AUC: [0.73562, 0.73972, 0.73762, 0.73616, 0.73826]
Fold AUC 평균±표준편차: 0.73748 ± 0.00147
전체 OOF AUC (rank-정규화 기준): 0.73748


## 6. test 예측 평균 + 저장

In [9]:
test_pred = np.mean(test_fold_preds, axis=0)  # 이미 fold별로 rank-정규화 했으므로 단순 평균 가능

BLEND_DIR = SHARED_DIR / "blend_cache"
BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "lgbm_lambdarank_oof.npy", oof)
np.save(BLEND_DIR / "lgbm_lambdarank_test.npy", test_pred)
print(f"저장 완료: {BLEND_DIR / 'lgbm_lambdarank_oof.npy'}, {BLEND_DIR / 'lgbm_lambdarank_test.npy'}")

# 실험 단계 비제출 CSV (제출 여부는 OOF 결과 보고 판단)
SUBMIT_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": test_pred})
out_path = SUBMIT_DIR / f"2차_lgbm_lambdarank_local{oof_auc:.5f}.csv"
submission.to_csv(out_path, index=False)
print(f"비제출용 결과 저장: {out_path}")


저장 완료: ../blend_cache/lgbm_lambdarank_oof.npy, ../blend_cache/lgbm_lambdarank_test.npy
비제출용 결과 저장: ../submit file/2차_lgbm_lambdarank_local0.73748.csv


## 7. 결과 해석 & 다음 단계

- **OOF AUC가 0.74037(멀티시드 배깅 최고치)보다 확실히 높은 group_size가 있다면**: 손실함수 변경이
  실제로 먹힌 것 → `blend_cache/lgbm_lambdarank_oof.npy`를 팀 블렌딩 노트북에 추가해볼 가치 있음.
- **모든 group_size에서 비슷하거나 낮다면(1차 결과 100=0.73599처럼)**: 인공 그룹 기반 lambdarank
  자체가 이 데이터셋/전처리에는 맞지 않는다는 결론 — 굳이 더 큰 그룹으로 추가 탐색할 필요 없이 폐기.
  (참고: group_size가 커질수록 fold_std가 줄어드는지도 같이 확인 — 줄어드는데 OOF는 여전히 낮다면
  "노이즈는 줄었지만 방법 자체의 천장이 낮다"는 뜻)
- XGBoost `rank:pairwise`도 같은 group 구성 + rank-정규화 패턴으로 시도 가능
  (기존 `xgboost_best_params.json` 재사용) — 단, lambdarank가 이미 분명히 나쁘다면 우선순위는 낮춤.
